# EFSM Phase 2 — Dataset Preparation

**Goal:** Download `facebook/empathetic_dialogues`, format as ChatML conversations, and save to `data/train.jsonl`, `data/val.jsonl`, `data/test.jsonl`.

**Expected outcomes:**
- `data/train.jsonl` ~17,000 rows (80% stratified split)
- `data/val.jsonl`   ~2,100 rows  (10%)
- `data/test.jsonl`  ~2,100 rows  (10%)
- Each row is a JSON array of ChatML message dicts with emotion-tagged user turns
- EFSMDataset label mask verified: only assistant tokens unmasked

---
**Before running:** No GPU needed — run on Kaggle CPU. Add `HF_TOKEN` as a Kaggle Secret.

## Cell 1 — Load Kaggle Secrets

In [21]:
from kaggle_secrets import UserSecretsClient
import os

secrets = UserSecretsClient()
os.environ['HF_TOKEN']      = secrets.get_secret('HF_TOKEN')
os.environ['WANDB_API_KEY'] = secrets.get_secret('WANDB_API_KEY')

print('Secrets loaded.')

Secrets loaded.


## Cell 2 — Clone Repo and Install Dependencies

In [22]:
!git clone https://github.com/tasbidrahman10/empathetic-voice-llm.git
%cd empathetic-voice-llm
!pip install -r requirements.txt -q

Cloning into 'empathetic-voice-llm'...
remote: Enumerating objects: 72, done.
remote: Counting objects: 100% (72/72), done.
remote: Compressing objects: 100% (41/41), done.
remote: Total 72 (delta 34), reused 58 (delta 20), pack-reused 0 (from 0)
Receiving objects: 100% (72/72), 61.94 KiB | 2.38 MiB/s, done.
Resolving deltas: 100% (34/34), done.
/kaggle/working/empathetic-voice-llm/empathetic-voice-llm/empathetic-voice-llm/empathetic-voice-llm/empathetic-voice-llm


In [23]:
import os
os.chdir('/kaggle/working/empathetic-voice-llm/empathetic-voice-llm/empathetic-voice-llm/empathetic-voice-llm')
print(os.getcwd())


/kaggle/working/empathetic-voice-llm/empathetic-voice-llm/empathetic-voice-llm/empathetic-voice-llm


## Cell 3 — Run Preprocessing

Downloads `facebook/empathetic_dialogues`, builds ChatML conversations, and saves the three JSONL splits.

In [24]:
!python src/data/preprocess_empathetic.py --config configs/config.yaml

Loading facebook/empathetic_dialogues (train split) ...
Raw utterance rows: 76673
Total conversations built: 17780

Train: 14224 conversations
  Avg turns per conversation: 4.3
  Emotion distribution (top 10):
    surprised                   739
    excited                     546
    angry                       509
    proud                       508
    annoyed                     489
    sad                         487
    grateful                    469
    lonely                      467
    afraid                      467
    terrified                   460

Val  : 1778 conversations
  Avg turns per conversation: 4.3
  Emotion distribution (top 10):
    surprised                    93
    excited                      68
    angry                        64
    proud                        63
    sad                          61
    annoyed                      61
    afraid                       59
    terrified                    58
    grateful                     58
    lonely  

## Cell 4 — Spot-Check: Display Sample Conversations

Load `data/train.jsonl` and print 3 formatted conversations to verify emotion tags and ChatML structure.

In [25]:
import json

TRAIN_PATH = 'data/train.jsonl'

with open(TRAIN_PATH) as f:
    samples = [json.loads(line) for line in f]

print(f'Total train conversations: {len(samples)}')
print()

for idx in [0, 500, 1000]:
    print(f'--- Conversation {idx} ---')
    for msg in samples[idx]:
        role = msg['role'].upper()
        content = msg['content']
        # Truncate long system prompt for display
        if role == 'SYSTEM':
            content = content[:80] + '...'
        print(f'[{role}] {content}')
    print()

Total train conversations: 14224

--- Conversation 0 ---
[SYSTEM] You are an empathetic voice therapist. Your role is to listen carefully to how t...
[USER] [emotion: embarrassed] I can't believe I actually went to work wearing my wife's dress the other day. I knew something was wrong when it felt like I was wrapped like a sausage. Luckily I had a spare suit in my office to change into.
[ASSISTANT] I can't believe I actually went to work wearing my wife's dress the other day. I knew something was wrong when it felt like I was wrapped like a sausage. Luckily I had a spare suit in my office to change into.
[USER] [emotion: embarrassed] I can't believe I actually went to work wearing my wife's dress the other day. I knew something was wrong when it felt like I was wrapped like a sausage. Luckily I had a spare suit in my office to change into.
[ASSISTANT] I can't believe I actually went to work wearing my wife's dress the other day. I knew something was wrong when it felt like I was wrappe

## Cell 5 — Verify EFSMDataset: Tokenisation and Label Masking

Loads the Qwen tokenizer (text-only, no model weights needed), instantiates `EFSMDataset`, and checks that the label mask correctly exposes only assistant-turn tokens.

In [26]:
import torch
from transformers import AutoTokenizer
from huggingface_hub import login
from src.data.dataset import EFSMDataset

login(token=os.environ['HF_TOKEN'])

MODEL_ID = 'Qwen/Qwen2.5-Omni-7B'
print('Loading tokenizer (no model weights downloaded) ...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=os.environ['HF_TOKEN'])

dataset = EFSMDataset('data/train.jsonl', tokenizer, max_seq_len=2048)
print(f'Dataset length: {len(dataset)}')

sample = dataset[0]
input_ids = sample['input_ids']
labels    = sample['labels']

unmasked = (labels != -100).sum().item()
total    = labels.shape[0]
masked   = total - unmasked

print(f'\nSample[0]:')
print(f'  input_ids shape : {input_ids.shape}')
print(f'  labels shape    : {labels.shape}')
print(f'  Masked tokens   : {masked} / {total}  ({100*masked/total:.1f}%)')
print(f'  Unmasked tokens : {unmasked} / {total}  ({100*unmasked/total:.1f}%)')
print()

# Decode the unmasked (assistant) tokens to verify they look correct
assistant_ids = input_ids[labels != -100]
assistant_text = tokenizer.decode(assistant_ids, skip_special_tokens=True)
print(f'Assistant tokens decoded (first 200 chars):')
print(assistant_text[:200])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Loading tokenizer (no model weights downloaded) ...
Dataset length: 14224

Sample[0]:
  input_ids shape : torch.Size([272])
  labels shape    : torch.Size([272])
  Masked tokens   : 174 / 272  (64.0%)
  Unmasked tokens : 98 / 272  (36.0%)

Assistant tokens decoded (first 200 chars):
I can't believe I actually went to work wearing my wife's dress the other day. I knew something was wrong when it felt like I was wrapped like a sausage. Luckily I had a spare suit in my office to cha


## Cell 6 — Upload JSONL Files to HuggingFace Hub

Backs up the three JSONL files to `{HF_USERNAME}/efsm-checkpoints` on HuggingFace so they persist across Kaggle sessions and can be downloaded for Phase 3 training.

In [27]:
from huggingface_hub import HfApi
import yaml

with open('configs/config.yaml') as f:
    cfg = yaml.safe_load(f)

# Derive repo id — replace with your HuggingFace username
HF_USERNAME = 'tasbid001'   # <-- update if your HF username differs
CHECKPOINT_REPO = 'tasbid001/efsm-checkpoints'


api = HfApi(token=os.environ['HF_TOKEN'])

for local_path in ['data/train.jsonl', 'data/val.jsonl', 'data/test.jsonl']:
    filename = local_path.split('/')[-1]
    api.upload_file(
        path_or_fileobj=local_path,
        path_in_repo=f'data/{filename}',
        repo_id=CHECKPOINT_REPO,
        repo_type='model',
        commit_message=f'Phase 2: upload {filename}',
    )
    print(f'Uploaded {local_path}  →  {CHECKPOINT_REPO}/data/{filename}')

print('\nPhase 2 COMPLETE. All JSONL files backed up to HuggingFace Hub.')

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Uploaded data/train.jsonl  →  tasbid001/efsm-checkpoints/data/train.jsonl
Uploaded data/val.jsonl  →  tasbid001/efsm-checkpoints/data/val.jsonl
Uploaded data/test.jsonl  →  tasbid001/efsm-checkpoints/data/test.jsonl

Phase 2 COMPLETE. All JSONL files backed up to HuggingFace Hub.
